> **Solutions for `4-hypothesis-testing-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 4: Hypothesis Testing

> **Course:** Data Science Foundations
> **Lesson:** Day 4 Lab, companion to `4-hypothesis-testing.md`
> **Dataset:** `penguins` (seaborn) plus simulated A/B experiments
> **Estimated time:** 2 hours (Parts 5 and 7 are extensions and can be skipped if time is short)

## Lab Objectives

- Frame real questions as $H_0$ / $H_1$ and pick an appropriate test
- Run a one-sample t-test **by hand** and confirm it with SciPy, including a **confidence interval**
- Run a **Welch** two-sample t-test, report an **effect size** (Cohen's d), and check the assumptions
- Use chi-square tests of independence and read the expected-count table
- Build a **permutation test** from scratch
- Show by simulation why **multiple testing** inflates false positives, and how Bonferroni fixes it
- Estimate the **sample size** an A/B test needs for 80% power

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
alpha = 0.05

penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print(f"Shape: {penguins.shape}")
penguins.head()

## Part 1: Framing the Question

For each scenario write: $H_0$, $H_1$, one- or two-tailed, the test you would use, and which error (Type I or II) is more costly.

1. A bakery claims its loaves weigh 500 g on average. A consumer group weighs 30 loaves.
2. An online shop tests a new checkout page against the old one and records whether each visitor buys.
3. A logistics firm compares average delivery times across **three** couriers.
4. A bank asks whether customers' preferred payment method (card, mobile money, cash) is related to their region.

**Model answer:**

| # | $H_0$ | $H_1$ | Tails | Test | Costlier error |
|---|---|---|---|---|---|
| 1 | Mean weight = 500 g | Mean weight < 500 g (the consumer worry) | One (left), chosen *before* weighing | One-sample t-test | Type II: underweight loaves go unchallenged. The bakery would argue Type I (a false accusation). |
| 2 | Conversion new = old | Conversion new ≠ old | Two | Two-proportion z-test (or chi-square on the 2×2 table) | Usually Type I: you ship a page that is not better and pay the rebuild cost for nothing |
| 3 | All three mean times are equal | At least one mean differs | n/a (F-test) | One-way ANOVA, then pairwise tests with a correction | Depends on the contract: dropping a good courier (Type I) vs keeping a slow one (Type II) |
| 4 | Payment method is independent of region | They are associated | n/a | Chi-square test of independence | Usually Type I: redesigning regional products around a pattern that is not real |

## Part 2: One-Sample t-Test by Hand

A field guide states that Adelie penguins have a mean flipper length of **190 mm**. Does our sample disagree?

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \qquad df = n - 1, \qquad \text{95\% CI} = \bar{x} \pm t_{0.975,\,df} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
adelie_flipper = penguins.loc[penguins["species"] == "Adelie", "flipper_length_mm"]
mu_0 = 190

n = len(adelie_flipper)
x_bar = adelie_flipper.mean()
s = adelie_flipper.std(ddof=1)
se = s / np.sqrt(n)
t_manual = (x_bar - mu_0) / se
df = n - 1
p_manual = 2 * stats.t.sf(abs(t_manual), df)
t_crit = stats.t.ppf(0.975, df)
ci_manual = (x_bar - t_crit * se, x_bar + t_crit * se)

res = stats.ttest_1samp(adelie_flipper, mu_0)

print(f"n = {n}, mean = {x_bar:.2f}, sd = {s:.2f}, SE = {se:.3f}")
print(f"t (manual) = {t_manual:.4f}   t (scipy) = {res.statistic:.4f}")
print(f"p (manual) = {p_manual:.4f}   p (scipy) = {res.pvalue:.4f}")
print(f"95% CI for the mean: ({ci_manual[0]:.2f}, {ci_manual[1]:.2f})")
print("Decision:", "reject H0" if p_manual <= alpha else "fail to reject H0")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(t_manual - res.statistic) < 1e-9 and abs(p_manual - res.pvalue) < 1e-9
ci = res.confidence_interval(0.95)
assert abs(ci_manual[0] - ci.low) < 1e-9 and abs(ci_manual[1] - ci.high) < 1e-9
print("passed")

**Question 2.** State your decision. Does it *prove* that the true mean is 190 mm? Re-run with `mu_0 = 192`. What changes, and how could you have predicted it from the confidence interval alone?

**Model answer:**

With $\mu_0 = 190$, p ≈ 0.85, so we **fail to reject** $H_0$. That does not prove the mean is 190. It only means the data is consistent with 190, and also with every other value inside the 95% CI (about 189.0 to 191.2 mm).

With $\mu_0 = 192$ we reject (p ≈ 0.0006). You can read that off the interval: a two-tailed test at α = 0.05 rejects exactly those $\mu_0$ values that fall **outside** the 95% CI. Confidence intervals and tests are two views of the same calculation, and the interval is usually more informative.

## Part 3: Welch Two-Sample t-Test, Effect Size, and Assumptions

**Question:** do male and female Gentoo penguins differ in body mass?

As the lesson explains, the t-statistic with separate variances, $\sqrt{s_A^2/n_A + s_B^2/n_B}$, is **Welch's** t-test. Remember that `stats.ttest_ind` defaults to Student's version, which assumes equal variances, so pass `equal_var=False`. Here you will also run Student's version to see when the two differ.

**Cohen's d** measures the size of the difference in standard-deviation units: $d = (\bar{x}_A - \bar{x}_B) / s_{pooled}$, where $s_{pooled} = \sqrt{\frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}}$. As a rough guide, 0.2 is small, 0.5 medium, and 0.8 large.

In [ ]:
gentoo = penguins[penguins["species"] == "Gentoo"]
male = gentoo.loc[gentoo["sex"] == "Male", "body_mass_g"]
female = gentoo.loc[gentoo["sex"] == "Female", "body_mass_g"]

welch = stats.ttest_ind(male, female, equal_var=False)
student = stats.ttest_ind(male, female)


def cohens_d(a, b):
    na, nb = len(a), len(b)
    s_pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / s_pooled


diff_ci = welch.confidence_interval(0.95)

print(f"Male   n={len(male)}, mean={male.mean():.0f} g, sd={male.std():.0f}")
print(f"Female n={len(female)}, mean={female.mean():.0f} g, sd={female.std():.0f}")
print(f"Welch   t = {welch.statistic:.2f}, p = {welch.pvalue:.2e}")
print(f"Student t = {student.statistic:.2f}, p = {student.pvalue:.2e}")
print(f"Difference = {male.mean() - female.mean():.0f} g, 95% CI ({diff_ci.low:.0f}, {diff_ci.high:.0f})")
print(f"Cohen's d = {cohens_d(male, female):.2f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert welch.pvalue < 1e-10
assert 2 < cohens_d(male, female) < 4
print("passed")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=gentoo, x="body_mass_g", hue="sex", kde=True, ax=ax)
ax.set_title("Gentoo body mass by sex")
plt.show()

print(f"Shapiro male:   p = {stats.shapiro(male).pvalue:.3f}")
print(f"Shapiro female: p = {stats.shapiro(female).pvalue:.3f}")
print(f"Mann-Whitney U: p = {stats.mannwhitneyu(male, female).pvalue:.2e}")

**Question 3.** Report the result the way you would in a short write-up: direction, size in grams, CI, effect size, p-value, and whether the assumptions look reasonable.

**Model answer:**

Male Gentoo penguins are heavier than females by about 805 g on average (95% CI roughly 700 to 910 g; Welch t ≈ 14.8, p < 0.001). Cohen's d ≈ 2.7 is a very large effect: the two distributions barely overlap. Shapiro tests give no strong evidence against Normality within each group, and the non-parametric Mann-Whitney test agrees (p < 0.001), so the conclusion does not depend on the Normality assumption. Student's and Welch's tests agree here because the group variances are similar. When variances or group sizes differ a lot, they can disagree, which is why Welch is the safer default.

## Part 4: Chi-Square Tests of Independence

The test compares observed counts with the counts **expected if the variables were independent**: $E_{ij} = \frac{\text{row total}_i \times \text{column total}_j}{N}$. It is only reliable when every expected count is at least about 5.

In [ ]:
for row, col in [("species", "island"), ("sex", "species")]:
    observed = pd.crosstab(penguins[row], penguins[col])
    chi2, p, dof, expected = stats.chi2_contingency(observed)
    print(f"===== {row} vs {col}")
    print("Observed:\n", observed)
    print("Expected under independence:\n", pd.DataFrame(expected, index=observed.index, columns=observed.columns).round(1))
    print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.3g}\n")

**Question 4.** Interpret both tests. For species vs island, what do the zeros in the observed table tell you that the p-value does not?

**Model answer:**

**Species vs island:** p is essentially 0, so they are strongly associated. The zeros tell the real story: Gentoo live only on Biscoe and Chinstrap only on Dream, while Adelie appear on all three islands. The p-value only says "not independent"; the table says *how*.

**Sex vs species:** p ≈ 0.98, so we fail to reject independence. The male/female split is close to 50/50 within every species, just as the expected table predicts.

## Part 5: More Than Two Groups: ANOVA (extension)

To compare three or more group means, use a one-way **ANOVA** (`stats.f_oneway`) instead of many t-tests. A significant ANOVA says *at least one* mean differs. Follow-up pairwise tests show which ones, but they need a multiple-testing correction (Part 7).

In [ ]:
groups = {sp: g["bill_length_mm"] for sp, g in penguins.groupby("species")}
anova = stats.f_oneway(*groups.values())
print(f"ANOVA: F = {anova.statistic:.1f}, p = {anova.pvalue:.2e}")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=penguins, x="species", y="bill_length_mm", ax=ax)
plt.show()

pairs = [("Adelie", "Chinstrap"), ("Adelie", "Gentoo"), ("Chinstrap", "Gentoo")]
for a, b in pairs:
    p = stats.ttest_ind(groups[a], groups[b], equal_var=False).pvalue
    print(f"{a:>9s} vs {b:<9s}  p = {p:.2e}  significant at alpha/3: {p <= alpha / 3}")

**Model answer:**

The ANOVA is overwhelming (F ≈ 400, p < 0.001), so bill length differs across species. Pairwise, Adelie differs sharply from both others (bills about 9 to 10 mm shorter). Chinstrap vs Gentoo is also significant even after Bonferroni, although the gap is small (about 1.3 mm). That is a reminder to report the size of each difference, not just the p-value.

## Part 6: A Permutation Test from Scratch

A permutation test builds the null distribution directly, with no Normality assumption. **If $H_0$ is true, the group labels are arbitrary.** So shuffle the labels many times, recompute the difference each time, and see how often a shuffled difference is at least as extreme as the real one.

Question: do male and female **Chinstrap** penguins differ in bill depth?

In [ ]:
chin = penguins[penguins["species"] == "Chinstrap"]
a = chin.loc[chin["sex"] == "Male", "bill_depth_mm"].to_numpy()
b = chin.loc[chin["sex"] == "Female", "bill_depth_mm"].to_numpy()


def permutation_test(a, b, n_perm=10_000, rng=rng):
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    null = np.empty(n_perm)
    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        null[i] = shuffled[:len(a)].mean() - shuffled[len(a):].mean()
    p_value = (np.sum(np.abs(null) >= abs(observed)) + 1) / (n_perm + 1)
    return observed, null, p_value


observed, null_diffs, p_perm = permutation_test(a, b)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(null_diffs, bins=50, ax=ax, color="grey")
ax.axvline(observed, color="red", lw=2, label=f"observed = {observed:.2f} mm")
ax.axvline(-observed, color="red", lw=2, ls="--")
ax.set_title("Null distribution from 10,000 label shuffles")
ax.legend()
plt.show()

print(f"Permutation p = {p_perm:.5f}")
print(f"Welch t-test p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.2e}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
obs, null, p = permutation_test(np.array([1.0, 2, 3]), np.array([1.0, 2, 3]), n_perm=200)
assert obs == 0 and p == 1.0 and len(null) == 200
print("passed")

**Model answer:**

The observed difference (males about 1.7 mm deeper) lies far outside all 10,000 shuffled differences, so the permutation p-value hits its floor of $1/10001 \approx 0.0001$. The Welch test agrees (p < 0.001). The permutation test cannot report anything smaller than $1/(n_{perm}+1)$. To resolve smaller p-values you need more permutations, but for a decision at α = 0.05 this is plenty.

## Part 7: The Multiple Testing Trap (extension)

Your team runs an A/A test (both groups get the *same* page, so $H_0$ is true for everything) and checks **20 different metrics**, each at α = 0.05. How often does *at least one* metric come out "significant"?

In [ ]:
n_experiments, n_metrics, n_per_group = 1000, 20, 50

group_a = rng.normal(0, 1, size=(n_experiments, n_metrics, n_per_group))
group_b = rng.normal(0, 1, size=(n_experiments, n_metrics, n_per_group))
p_values = stats.ttest_ind(group_a, group_b, axis=-1).pvalue

any_hit = np.mean((p_values <= alpha).any(axis=1))
any_hit_bonf = np.mean((p_values <= alpha / n_metrics).any(axis=1))
print(f"At least one false positive (alpha = 0.05):       {any_hit:.3f}   theory {1 - 0.95 ** 20:.3f}")
print(f"At least one false positive (Bonferroni 0.0025):  {any_hit_bonf:.3f}   theory {1 - (1 - 0.0025) ** 20:.3f}")

**Model answer:**

About 62% of pure-noise experiments produce at least one "significant" metric, close to the theoretical $1 - 0.95^{20} \approx 0.64$. Each test has a 5% false-positive rate, but the chance that *some* test fires grows quickly with the number of tests. Bonferroni divides α by the number of tests and brings the family-wise rate back to about 5%. It is conservative, but simple and valid. Decide your primary metric *before* the experiment, and correct for the rest.

## Part 8: How Big Should the A/B Test Be?

The lesson's simulation found that 1,000 users per group gives only modest power to detect a lift from 10% to 12%. Find the sample size per group that gives **80% power**.

Planning formula for comparison: $n \approx \dfrac{(z_{1-\alpha/2} + z_{0.8})^2 \,[p_A(1-p_A) + p_B(1-p_B)]}{(p_B - p_A)^2}$

In [ ]:
p_A, p_B, n_sims = 0.10, 0.12, 4000
sizes = [500, 1000, 2000, 3000, 4000, 5000, 6000]


def simulated_power(n, p_A, p_B, n_sims, rng=rng):
    conv_A = rng.binomial(n, p_A, n_sims)
    conv_B = rng.binomial(n, p_B, n_sims)
    pooled = (conv_A + conv_B) / (2 * n)
    se = np.sqrt(pooled * (1 - pooled) * (2 / n))
    z = (conv_B / n - conv_A / n) / se
    p = 2 * stats.norm.sf(np.abs(z))
    return np.mean(p <= alpha)


powers = pd.Series({n: simulated_power(n, p_A, p_B, n_sims) for n in sizes}, name="power")
display(powers.round(3))

z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(0.8)
n_formula = (z_a + z_b) ** 2 * (p_A * (1 - p_A) + p_B * (1 - p_B)) / (p_B - p_A) ** 2
print(f"Formula: about {np.ceil(n_formula):.0f} users per group")

fig, ax = plt.subplots(figsize=(8, 4))
powers.plot(marker="o", ax=ax)
ax.axhline(0.8, color="red", ls="--", label="80% power")
ax.axvline(n_formula, color="grey", ls=":", label=f"formula n ≈ {n_formula:.0f}")
ax.set_xlabel("users per group")
ax.set_ylabel("power")
ax.legend()
plt.show()

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert simulated_power(1000, 0.10, 0.10, 4000) < 0.08   # with no real effect, "power" is just alpha
assert simulated_power(6000, 0.10, 0.12, 4000) > 0.8
print("passed")

**Question 8.** A manager wants to run the test for one week with 1,000 users per group "to be quick". Using your results, explain the risk in two sentences.

**Model answer:**

With 1,000 users per group, the test has only about 30% power: even if the new page truly lifts conversion from 10% to 12%, we would miss it about 70% of the time and probably scrap a winning design. Detecting a 2-point lift reliably needs about 3,800 users per group, so the test should run roughly four times longer, or target a bigger effect.

## Part 9: Wrap-Up

Name three things from today that the p-value alone would not have told you.

**Model answer:**

1. **How big the effect is.** Gentoo males are about 805 g heavier (d ≈ 2.7). A tiny p-value can come from a trivial difference in a big sample.
2. **The plausible range.** The confidence interval shows which values are consistent with the data. "Fail to reject 190 mm" does not mean "it is 190 mm".
3. **Whether the test was fair.** Assumptions (Normality, equal variances, expected counts ≥ 5), the number of tests you ran (multiple testing), and whether the sample was large enough (power) all determine whether the p-value means anything.

> Submit your completed notebook to the Kanban board under your name by end of day.